# Best Model — SVC Aggressive

LinearSVC + OneVsRest with word uni/bi TF-IDF + char 3-gram TF-IDF + Word2Vec features.

**Posts F1=0.808 | Comments F1=0.703**

## 1. Setup & Imports

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    Tokenizer, StopWordsRemover, NGram, RegexTokenizer,
    CountVectorizer, IDF, Word2Vec, VectorAssembler, StringIndexer,
)
from pyspark.ml.classification import LinearSVC, OneVsRest
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

spark = SparkSession.builder \
    .appName('BestModel_SVC_Aggressive') \
    .getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} - ready')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/28 12:30:45 INFO SparkEnv: Registering MapOutputTracker
26/04/28 12:30:45 INFO SparkEnv: Registering BlockManagerMaster
26/04/28 12:30:45 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/04/28 12:30:45 INFO SparkEnv: Registering OutputCommitCoordinator


Spark 3.5.3 - ready


## 2. Load Data

In [ ]:
posts_df    = spark.read.parquet('gs://reddit-ai-2/process_data/Sentiment_Train_DataSet/sentiment_analysis_roberta_POST.parquet')
comments_df = spark.read.parquet('gs://reddit-ai-2/process_data/Sentiment_Train_DataSet/sentiment_analysis_roberta_COMMENT.parquet')

print(f'Posts:    {posts_df.count():,} rows')
print(f'Comments: {comments_df.count():,} rows')

posts_df_rest = spark.read.parquet('gs://reddit-ai-2/process_data/Post_Single_or_Non_Tag/post_with_topic.parquet/part*')
comment_df_rest = spark.read.parquet('gs://reddit-ai-2/process_data/Comment_Simple_or_non_tag/comment_with_topic.parquet/part*')


print(f'Posts:    {posts_df_rest.count():,} rows')
print(f'Comments: {comment_df_rest.count():,} rows')

Posts:    55,147 rows
Comments: 55,216 rows


Posts:    255,607 rows


Comments: 3,739,964 rows


## 3. Clean Text + Label Distribution

In [ ]:
def clean_text(col):
    c = F.coalesce(col, F.lit(''))
    c = F.regexp_replace(c, r'http\S+|www\.\S+', ' ')
    c = F.regexp_replace(c, r'\[deleted\]|\[removed\]', ' ')
    c = F.regexp_replace(c, r'&amp;|&gt;|&lt;|&#x200B;', ' ')
    c = F.regexp_replace(c, r'[^\x00-\x7F]+', ' ')
    c = F.regexp_replace(c, r'[\r\n\t]+', ' ')
    c = F.regexp_replace(c, r'\s+', ' ')
    return F.lower(F.trim(c))

MIN_LEN = 3

posts_clean_df = (
    posts_df
    .withColumn('title_clean', clean_text(F.col('title')))
    .filter(F.length('title_clean') >= MIN_LEN)
    .select('title_clean', 'sentiment_label')
    .cache()
)

comments_clean_df = (
    comments_df
    .withColumn('body_clean', clean_text(F.col('body')))
    .filter(F.length('body_clean') >= MIN_LEN)
    .select('body_clean', 'sentiment_label')
    .cache()
)

posts_df_rest = (
    posts_df_rest
    .withColumn('title_clean', clean_text(F.col('title')))
    .filter(F.length('title_clean') >= MIN_LEN)
    .cache()
)

comment_df_rest = (
    comment_df_rest
    .withColumn('body_clean', clean_text(F.col('body')))
    .filter(F.length('body_clean') >= MIN_LEN)
    .cache()
)


print(f'Posts after clean:    {posts_clean_df.count():,} rows')
print(f'Comments after clean: {comments_clean_df.count():,} rows')
print(f'Posts after clean:    {posts_df_rest.count():,} rows')
print(f'Comments after clean: {comment_df_rest.count():,} rows')


print('\nPosts - Label Distribution:')
posts_clean_df.groupBy('sentiment_label').count().orderBy('sentiment_label').show()

print('Comments - Label Distribution:')
comments_clean_df.groupBy('sentiment_label').count().orderBy('sentiment_label').show()

Posts after clean:    54,835 rows


Comments after clean: 54,609 rows


Posts after clean:    254,073 rows


Comments after clean: 3,699,717 rows

Posts - Label Distribution:


+---------------+-----+
|sentiment_label|count|
+---------------+-----+
|        LABEL_0|12273|
|        LABEL_1|37377|
|        LABEL_2| 5185|
+---------------+-----+

Comments - Label Distribution:
+---------------+-----+
|sentiment_label|count|
+---------------+-----+
|        LABEL_0|16518|
|        LABEL_1|27662|
|        LABEL_2|10429|
+---------------+-----+



## 4. Build Aggressive Pipeline

In [ ]:
english_stopwords = StopWordsRemover.loadDefaultStopWords('english')

def build_pipeline(text_col):
    # word path
    tok_w  = Tokenizer(inputCol=text_col, outputCol='tokens')
    sw     = StopWordsRemover(inputCol='tokens', outputCol='tokens_filtered',
                              stopWords=english_stopwords)
    ng_w   = NGram(n=2, inputCol='tokens_filtered', outputCol='bigrams')
    cv_uni = CountVectorizer(inputCol='tokens_filtered', outputCol='tf_uni',
                             vocabSize=15000, minDF=2.0, maxDF=0.95)
    cv_bi  = CountVectorizer(inputCol='bigrams', outputCol='tf_bi',
                             vocabSize=15000, minDF=2.0, maxDF=0.95)
    idf_u  = IDF(inputCol='tf_uni', outputCol='tfidf_uni')
    idf_b  = IDF(inputCol='tf_bi',  outputCol='tfidf_bi')

    # char path
    tok_c   = RegexTokenizer(inputCol=text_col, outputCol='chars',
                             pattern='.', gaps=False, toLowercase=False)
    ng_c    = NGram(n=3, inputCol='chars', outputCol='char_3grams')
    cv_char = CountVectorizer(inputCol='char_3grams', outputCol='tf_char',
                              vocabSize=10000, minDF=5.0, maxDF=0.95)
    idf_c   = IDF(inputCol='tf_char', outputCol='tfidf_char')

    # Word2Vec
    w2v = Word2Vec(inputCol='tokens_filtered', outputCol='w2v_vec',
                   vectorSize=50, minCount=2, numPartitions=4,
                   maxIter=5, seed=42)

    assembler = VectorAssembler(
        inputCols=['tfidf_uni', 'tfidf_bi', 'tfidf_char', 'w2v_vec'],
        outputCol='features',
    )

    return Pipeline(stages=[
        tok_w, sw, ng_w, cv_uni, cv_bi, idf_u, idf_b,
        tok_c, ng_c, cv_char, idf_c,
        w2v, assembler
    ])

print('Pipeline builder ready.')

Pipeline builder ready.


In [ ]:
# String Indexer Convert Label_0 (Negative) to 1, Label_1 (Neutral) to 0 and Label_2 (Positive) not change
indexer = StringIndexer(inputCol='sentiment_label', outputCol='label')

svc = LinearSVC(featuresCol='features', labelCol='label',
                    maxIter=30, regParam=0.01)
ovr = OneVsRest(classifier=svc, featuresCol='features', labelCol='label')

## 5. Train & Evaluate — Posts

In [ ]:
acc_eval = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='accuracy')
f1_eval  = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='f1')

print('Training Posts model...')
posts_train, posts_test = posts_clean_df.randomSplit([0.8, 0.2], seed=42)

posts_train.show(3)

# Build and fit the feature engineering pipeline
posts_pipeline_model = build_pipeline('title_clean').fit(posts_train)

# Transform the training and test data to get features
posts_train = posts_pipeline_model.transform(posts_train)
posts_test = posts_pipeline_model.transform(posts_test)

# Index the sentiment labels
posts_train = indexer.fit(posts_train).transform(posts_train)
posts_test = indexer.fit(posts_test).transform(posts_test)

# Fit Train Model (OneVsRest with LinearSVC)
posts_model = ovr.fit(posts_train)

# Make predictions on the test set
posts_pred  = posts_model.transform(posts_test)

posts_acc = acc_eval.evaluate(posts_pred)
posts_f1  = f1_eval.evaluate(posts_pred)
print(f'Posts   - Accuracy: {posts_acc:.4f}  F1: {posts_f1:.4f}')

Training Posts model...
+--------------------+---------------+
|         title_clean|sentiment_label|
+--------------------+---------------+
|        !!! help !!!|        LABEL_1|
|              !!!!!!|        LABEL_1|
|" ", it's not cha...|        LABEL_1|
+--------------------+---------------+
only showing top 3 rows



26/04/28 12:36:11 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/04/28 12:36:18 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/04/28 12:36:20 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/04/28 12:36:22 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/04/28 12:36:22 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/04/28 12:36:23 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/04/28 12:36:23 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/04/28 12:36:24 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/04/28 12:36:24 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/04/28 12:36:25 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/04/28 12:36:25 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/04/28 12:36:26 WARN DAGScheduler: Broadcasting larg

26/04/28 12:37:05 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/04/28 12:37:06 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/04/28 12:37:06 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/04/28 12:37:06 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/04/28 12:37:07 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/04/28 12:37:07 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/04/28 12:37:08 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/04/28 12:37:08 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/04/28 12:37:08 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/04/28 12:37:09 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/04/28 12:37:09 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/04/28 12:37:09 WARN DAGScheduler: Broadcasting larg

26/04/28 12:37:42 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/04/28 12:37:46 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB
26/04/28 12:37:55 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB


Posts   - Accuracy: 0.8017  F1: 0.7977


## 6. Train & Evaluate — Comments

In [ ]:
print('Training Comments model...')
comments_train, comments_test = comments_clean_df.randomSplit([0.8, 0.2], seed=42)

# Build and fit the feature engineering pipeline
comments_pipeline_model = build_pipeline('body_clean').fit(comments_train)

# Transform the training and test data to get features
comments_train = comments_pipeline_model.transform(comments_train)
comments_test = comments_pipeline_model.transform(comments_test)

# Index the sentiment labels
comments_train = indexer.fit(comments_train).transform(comments_train)
comments_test = indexer.fit(comments_test).transform(comments_test)

# Fit Train Model (OneVsRest with LinearSVC)
comments_model = ovr.fit(comments_train)

# Make predictions on the test set
comments_pred  = comments_model.transform(comments_test)

comments_acc = acc_eval.evaluate(comments_pred)
comments_f1  = f1_eval.evaluate(comments_pred)
print(f'Comments - Accuracy: {comments_acc:.4f}  F1: {comments_f1:.4f}')

Training Comments model...


26/04/28 12:40:23 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/04/28 12:40:32 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/04/28 12:40:33 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/04/28 12:40:34 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/04/28 12:40:34 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/04/28 12:40:35 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/04/28 12:40:35 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/04/28 12:40:35 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/04/28 12:40:36 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/04/28 12:40:36 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/04/28 12:40:36 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/04/28 12:40:37 WARN DAGScheduler: Broadcasting larg

26/04/28 12:41:08 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/04/28 12:41:08 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/04/28 12:41:08 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/04/28 12:41:09 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/04/28 12:41:09 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/04/28 12:41:09 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/04/28 12:41:10 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/04/28 12:41:10 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/04/28 12:41:10 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/04/28 12:41:11 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/04/28 12:41:11 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/04/28 12:41:12 WARN DAGScheduler: Broadcasting larg

26/04/28 12:41:51 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB


Comments - Accuracy: 0.7112  F1: 0.7094


## 9. Deploy to Post Data

In [ ]:
posts_df_rest = posts_pipeline_model.transform(posts_df_rest)

posts_df_rest = posts_df_rest.drop('tokens','tokens_filtered','bigrams','tf_uni','tf_bi','tfidf_uni','tfidf_bi',
                                   'chars','char_3grams','tf_char','tfidf_char','w2v_vec')

posts_df_rest.show(3)

posts_df_rest = posts_model.transform(posts_df_rest)

+------------------+-------------------+------------------+----------------+------------+--------------+--------------------+---------+--------------------+------------------+---+-----+----------+-------+--------------------+--------------------+--------------------+
|content_categories|        created_utc|            author|is_crosspostable|num_comments|num_crossposts|            selftext|subreddit|               title|      upvote_ratio|ups|downs|view_count|AI_Name|     detected_topics|         title_clean|            features|
+------------------+-------------------+------------------+----------------+------------+--------------+--------------------+---------+--------------------+------------------+---+-----+----------+-------+--------------------+--------------------+--------------------+
|              NULL|2025-12-07 15:50:47|        LancerNerd|            true|           1|             0|                    |  ChatGPT|here is my chatgp...|0.7200000286102295|  3|    0|      NULL|

In [ ]:
posts_df_rest.show(3)

26/04/28 12:43:15 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB
26/04/28 12:43:17 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB


+------------------+-------------------+------------------+----------------+------------+--------------+--------------------+---------+--------------------+------------------+---+-----+----------+-------+--------------------+--------------------+--------------------+--------------------+----------+
|content_categories|        created_utc|            author|is_crosspostable|num_comments|num_crossposts|            selftext|subreddit|               title|      upvote_ratio|ups|downs|view_count|AI_Name|     detected_topics|         title_clean|            features|       rawPrediction|prediction|
+------------------+-------------------+------------------+----------------+------------+--------------+--------------------+---------+--------------------+------------------+---+-----+----------+-------+--------------------+--------------------+--------------------+--------------------+----------+
|              NULL|2025-12-07 15:50:47|        LancerNerd|            true|           1|           

In [ ]:
posts_df_rest = posts_df_rest.drop('features')

## 9. Deploy to Comment Data

In [ ]:
comment_df_rest = comments_pipeline_model.transform(comment_df_rest)

comment_df_rest = comment_df_rest.drop('tokens','tokens_filtered','bigrams','tf_uni','tf_bi','tfidf_uni','tfidf_bi',
                                   'chars','char_3grams','tf_char','tfidf_char','w2v_vec')

comment_df_rest = comments_model.transform(comment_df_rest)

comment_df_rest = comment_df_rest.drop('features')

comment_df_rest.show(3)

26/04/28 12:44:23 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
26/04/28 12:44:24 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB


+--------------------+-------------------+------------+-------------------+--------------+---------+-----+---+-------+------------------+--------------------+--------------------+----------+
|                body|        created_utc|is_submitter|             author|removal_reason|subreddit|score|ups|AI_Name|   detected_topics|          body_clean|       rawPrediction|prediction|
+--------------------+-------------------+------------+-------------------+--------------+---------+-----+---+-------+------------------+--------------------+--------------------+----------+
|so we’re just giv...|2026-02-11 07:50:17|       false|      WhipItWhippet|          NULL|  ChatGPT|    1|  1|ChatGPT|                []|so we re just giv...|[-0.4248796518462...|       1.0|
|this sounds inter...|2026-02-11 07:50:21|       false|           Pijlpunt|          NULL|  ChatGPT|    3|  3|ChatGPT|[usecase_creative]|this sounds inter...|[-4.1275340423823...|       2.0|
|not true. chatgpt...|2026-02-11 07:51:15|   

In [ ]:
comment_df_rest.select('rawPrediction','prediction').show(3)

26/04/28 12:45:35 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB
26/04/28 12:45:37 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB


+--------------------+----------+
|       rawPrediction|prediction|
+--------------------+----------+
|[-0.4248796518462...|       1.0|
|[-4.1275340423823...|       2.0|
|[2.10839264324635...|       0.0|
+--------------------+----------+
only showing top 3 rows



## 9. Write File

In [ ]:
target_path_parquet = "gs://reddit-ai-2/process_data/Post_Single_or_Non_Tag/post_sentimented.parquet"

posts_df_rest.write.mode("overwrite").parquet(target_path_parquet)

print(f"บันทึกไฟล์ post_sentimented.parquet เรียบร้อยแล้ว")

target_path_parquet = "gs://reddit-ai-2/process_data/Comment_Simple_or_non_tag/comment_sentimented.parquet"

comment_df_rest.write.mode("overwrite").parquet(target_path_parquet)

print(f"บันทึกไฟล์ comment_sentimented เรียบร้อยแล้ว")

26/04/28 12:49:45 WARN DAGScheduler: Broadcasting large task binary with size 3.7 MiB


บันทึกไฟล์ post_sentimented.parquet เรียบร้อยแล้ว


26/04/28 12:50:30 WARN DAGScheduler: Broadcasting large task binary with size 4.5 MiB


บันทึกไฟล์ comment_sentimented เรียบร้อยแล้ว


## 10. Create Sample File

In [ ]:
from pyspark.sql import functions as F

# 1. Add a unique ID to each row so we can track what was selected
posts_df_rest = posts_df_rest.withColumn("row_id", F.monotonically_increasing_id())
comment_df_rest = comment_df_rest.withColumn("row_id", F.monotonically_increasing_id())

# 2. Sampling
# Note: sample() is probabilistic, so we use limit(100000) to get exactly that amount
sampling_post = posts_df_rest.sample(withReplacement=False, fraction=0.01, seed=42).limit(100000)

sampling_comment = comment_df_rest.sample(withReplacement=False, fraction=0.01, seed=42).limit(100000)

# 3.Write File
target_path_parquet = "gs://reddit-ai-2/process_data/Sampling_Topic/sampling_post.parquet"

sampling_post.write.mode("overwrite").parquet(target_path_parquet)

target_path_parquet = "gs://reddit-ai-2/process_data/Sampling_Topic/sampling_comment.parquet"

sampling_comment.write.mode("overwrite").parquet(target_path_parquet)

26/04/28 13:00:19 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB
26/04/28 13:00:57 WARN DAGScheduler: Broadcasting large task binary with size 4.2 MiB


## 11. Stop Spark

In [ ]:
spark.stop()
print('SparkSession stopped.')

SparkSession stopped.
